In [ ]:
import csv
import json
import pandas as pd
import numpy as np
from pydantic import ValidationError,field_validator
from datetime import datetime

In [ ]:
import os

os.makedirs("data", exist_ok=True)

In [ ]:
def parse_row(row):
    parsed = row.copy()
    Schema = {
        "age":int,
        "fee_inr":int,
        "prior_no_shows":int,
        "distance_km":float
    }
    
    for column,data_type in Schema.items():
        try:
            parsed[column] = data_type(parsed[column])
        except ValueError as e:
            return None,column,e

    return parsed,None,None

    

In [ ]:
with open("data/appointment.csv", newline="") as appointment_csv:
    reader = csv.DictReader(appointment_csv)
    headers = reader.fieldnames    
    print(f"Header: {headers}")
    rejected_rows = []
    active_rows = []
    for row in reader:
        result,column,err = parse_row(row)       
        if result is None:            
            #rejected_rows.append({"row":row,"failed_column":column,"Error":err})
            rejected_rows.append(row)
        else:
            active_rows.append(result)

    if headers and "appointment_id" in headers:
        print(f"Total Rows:{reader.line_num-1}")
    else:
        print(f"Total Rows:{reader.line_num}")

    print(len(rejected_rows))


In [ ]:
with open("data/rejected_rows.json", "w", encoding="utf-8") as f:
  json.dump(rejected_rows, f, indent=2)

In [ ]:
df = pd.read_csv("data/appointment.csv")


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
print("Null counts:\n", df.isnull().sum())

In [ ]:
print("Duplicate count:", df.duplicated().sum())

In [ ]:
df = df.drop_duplicates()

In [ ]:
print("After dropping duplicates:", len(df))

In [ ]:
df["booking_channel"] = df["booking_channel"].str.strip().str.title()

In [ ]:
df['appt_date'] = pd.to_datetime(df["appt_date"],errors="coerce")
df['booked_on'] = pd.to_datetime(df["booked_on"],errors="coerce")

In [ ]:
df = df.dropna(subset=["appt_date", "status"])
print("After dropping missing dates/status:", len(df))

In [ ]:
df["age"] = pd.to_numeric(df["age"], errors="coerce")

In [ ]:
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan

In [ ]:
median_age = df["age"].median()
print("Calculated Median Age:", median_age)

In [ ]:
median_distance = df["distance_km"].median()
print(median_distance)

In [ ]:
df["age"] = df["age"].fillna(median_age).round().astype(int)


In [ ]:
df["distance_km"] = df["distance_km"].fillna(median_distance)

In [ ]:
for col in ["distance_km", "fee_inr"]:
  q1 = df[col].quantile(0.25)
  q3 = df[col].quantile(0.75)

  print(f"Q1 values in {col}: {q1}")
  print(f"Q3 values in {col}: {q3}")

  iqr = q3 - q1
  lower_fence = q1 - 1.5 * iqr
  upper_fence = q3 + 1.5 * iqr

  print(f"Lower fence in {col}: {lower_fence}")
  print(f"Upper fence in {col}: {upper_fence}")

  capped_count = ((df[col] < lower_fence) | (df[col] > upper_fence)).sum()
  print(f"Capped values in {col}: {capped_count}")
  df[col] = np.clip(df[col], lower_fence, upper_fence)
  

In [ ]:
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)